In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import sys
import os
sys.path.append(os.getcwd())
sys.path.append(os.path.dirname(os.getcwd()))
from src.train_pipeline import build_model_pipeline

In [3]:
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.ensemble import RandomForestClassifier

import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score

In [4]:
df = pd.read_csv(
    "../data/processed/final_training_data.csv"
)

X = df.drop(
    columns=["is_high_risk"]
)

y = df["is_high_risk"]

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
)

print(X_train.shape)
print(X_test.shape)

(76529, 34)
(19133, 34)


In [5]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5]
}

In [6]:
grid_search = GridSearchCV(
    RandomForestClassifier(
        random_state=42
    ),
    param_grid=param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(
    X_train,
    y_train
)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


GridSearchCV(cv=3, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, 20, None],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             scoring='roc_auc', verbose=2)

In [7]:
print(grid_search.best_params_)

{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}


In [8]:
hasattr(grid_search, "best_params_")

True

In [9]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest ROC-AUC:")
print(grid_search.best_score_)

Best Parameters:
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}

Best ROC-AUC:
0.9981601688619971


In [10]:
best_model = grid_search.best_estimator_

with mlflow.start_run(
    run_name="RandomForest_Tuned"
):

    mlflow.log_params(
        grid_search.best_params_
    )

    y_prob = best_model.predict_proba(
        X_test
    )[:, 1]

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    mlflow.log_metric(
        "roc_auc",
        roc_auc
    )

    mlflow.sklearn.log_model(
        best_model,
        "RandomForest_Tuned"
    )

print("Model logged successfully")

Model logged successfully


In [11]:
print(type(best_model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [12]:
import joblib

In [13]:
joblib.dump(
    best_model,
    "../models/random_forest.pkl"
)

['../models/random_forest.pkl']

In [14]:
import os

os.path.exists(
    "../models/random_forest.pkl"
)

True

In [16]:
import joblib

model = joblib.load("../models/random_forest.pkl")

print(type(model))
print("Features expected:", model.n_features_in_)

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
Features expected: 34


In [2]:
import os

print(
    os.path.exists(
        "../models/random_forest.pkl"
    )
)

True


In [4]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/final_training_data.csv"
)

print(df.columns.tolist())
print(len(df.columns))

['num__Amount', 'num__Value', 'num__transaction_hour', 'num__transaction_day', 'num__transaction_month', 'num__transaction_year', 'num__total_transaction_amount', 'num__avg_transaction_amount', 'num__transaction_count', 'num__std_transaction_amount', 'cat__CurrencyCode_UGX', 'cat__ProviderId_ProviderId_1', 'cat__ProviderId_ProviderId_2', 'cat__ProviderId_ProviderId_3', 'cat__ProviderId_ProviderId_4', 'cat__ProviderId_ProviderId_5', 'cat__ProviderId_ProviderId_6', 'cat__ProductCategory_airtime', 'cat__ProductCategory_data_bundles', 'cat__ProductCategory_financial_services', 'cat__ProductCategory_movies', 'cat__ProductCategory_other', 'cat__ProductCategory_ticket', 'cat__ProductCategory_transport', 'cat__ProductCategory_tv', 'cat__ProductCategory_utility_bill', 'cat__ChannelId_ChannelId_1', 'cat__ChannelId_ChannelId_2', 'cat__ChannelId_ChannelId_3', 'cat__ChannelId_ChannelId_5', 'cat__PricingStrategy_0', 'cat__PricingStrategy_1', 'cat__PricingStrategy_2', 'cat__PricingStrategy_4', 'is_hi

In [5]:
X = df.drop(
    columns=["is_high_risk"]
)

print(X.columns.tolist())

['num__Amount', 'num__Value', 'num__transaction_hour', 'num__transaction_day', 'num__transaction_month', 'num__transaction_year', 'num__total_transaction_amount', 'num__avg_transaction_amount', 'num__transaction_count', 'num__std_transaction_amount', 'cat__CurrencyCode_UGX', 'cat__ProviderId_ProviderId_1', 'cat__ProviderId_ProviderId_2', 'cat__ProviderId_ProviderId_3', 'cat__ProviderId_ProviderId_4', 'cat__ProviderId_ProviderId_5', 'cat__ProviderId_ProviderId_6', 'cat__ProductCategory_airtime', 'cat__ProductCategory_data_bundles', 'cat__ProductCategory_financial_services', 'cat__ProductCategory_movies', 'cat__ProductCategory_other', 'cat__ProductCategory_ticket', 'cat__ProductCategory_transport', 'cat__ProductCategory_tv', 'cat__ProductCategory_utility_bill', 'cat__ChannelId_ChannelId_1', 'cat__ChannelId_ChannelId_2', 'cat__ChannelId_ChannelId_3', 'cat__ChannelId_ChannelId_5', 'cat__PricingStrategy_0', 'cat__PricingStrategy_1', 'cat__PricingStrategy_2', 'cat__PricingStrategy_4']


In [9]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split

from src.train_pipeline import (
    build_model_pipeline
)

from src.default_estimator import (
    create_transaction_level_target
)

In [10]:
df = pd.read_csv(
    "../data/raw/data.csv"
)

In [11]:
df_target = create_transaction_level_target(
    df
)

In [12]:
X = df_target.drop(
    columns=["is_high_risk"]
)

y = df_target["is_high_risk"]

In [13]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
)

In [14]:
pipeline = build_model_pipeline()

In [15]:
pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('aggregate_features', AggregateFeatures()),
                                 ('datetime_features', DateTimeFeatures()),
                                 ('drop_columns',
                                  DropColumns(columns=['TransactionId',
                                                       'BatchId', 'AccountId',
                                                       'SubscriptionId',
                                                       'CustomerId',
                                                       'ProductId',
                                                       'TransactionStartTime'])),
                                 ('preprocessor',
                                  ColumnTransformer(sparse_threshold=0,
                                                    transformers=[('num',
                                                                   P...
                                                                    'avg_transaction_amount',
                                                                    'transaction_count',
                                                                    'std_transaction_amount']),
                                                                  ('cat',
                                                                   Pipeline(steps=[('imputer',
                                                                                    SimpleImputer(strategy='most_frequent')),
                                                                                   ('encoder',
                                                                                    OneHotEncoder(handle_unknown='ignore',
                                                                                                  sparse_output=False))]),
                                                                   ['CurrencyCode',
                                                                    'ProviderId',
                                                                    'ProductCategory',
                                                                    'ChannelId',
                                                                    'PricingStrategy'])]))])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [16]:
pipeline.score(
    X_test,
    y_test
)

0.873516960225788

In [17]:
joblib.dump(
    pipeline,
    "../models/credit_risk_pipeline.pkl"
)

['../models/credit_risk_pipeline.pkl']

In [18]:
import os

os.path.exists(
    "../models/credit_risk_pipeline.pkl"
)

True